# Multi-resolution Event V2 — relational primary

Select T4 x2, keep Internet enabled, and choose **Save & Run All**. Do not attach any Kaggle Input. This Notebook performs immediate GPU and private-Dataset access checks, downloads the single frozen r9 run bundle, verifies its identities, and launches the 47,801,855-parameter relational six-M4 primary. It does not clone source, rebuild data, run a disposable capacity model, or authorize block/trajectory ablations.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

DATASET_REF = 'vanilaaaa/trauma-predict-relational-primary-r9-bundle'
SCHEMA = 'trauma_predict.multires_event_v2_relational_primary_bundle.v2'
DOWNLOAD_ROOT = Path('/kaggle/working/relational_primary_r9_bundle_download')
PACKAGE_ROOT = DOWNLOAD_ROOT / 'dataset-package'

def run(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    subprocess.run(command, check=True)

def configure_kaggle_credentials():
    if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
        return
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        username = client.get_secret('KAGGLE_USERNAME')
        key = client.get_secret('KAGGLE_KEY')
    except Exception:
        return
    if username and key:
        os.environ['KAGGLE_USERNAME'] = username
        os.environ['KAGGLE_KEY'] = key

def find_bundle(root):
    matches = []
    for manifest_path in sorted(root.rglob('run_bundle_manifest.json')):
        payload = json.loads(manifest_path.read_text(encoding='utf-8'))
        if payload.get('schema') == SCHEMA:
            matches.append(manifest_path.parent)
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one relational-primary bundle under {root}, found {len(matches)}')
    return matches[0]

def extract_outer_package(archive, destination):
    destination.mkdir(parents=True, exist_ok=False)
    root = destination.resolve()
    with zipfile.ZipFile(archive) as handle:
        for info in handle.infolist():
            target = (destination / info.filename).resolve()
            try:
                target.relative_to(root)
            except ValueError as exc:
                raise ValueError(f'Dataset ZIP path escapes destination: {info.filename}') from exc
        handle.extractall(destination)

import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError(f'Select Kaggle T4 x2; found {torch.cuda.device_count()} visible GPU(s)')
print('RELATIONAL_PRIMARY_GPU_PREFLIGHT_OK GPUs=2', flush=True)
configure_kaggle_credentials()
run(['kaggle', 'datasets', 'files', '-d', DATASET_REF, '--page-size', '1'])
print('RELATIONAL_PRIMARY_DATASET_ACCESS_OK', DATASET_REF, flush=True)
if PACKAGE_ROOT.is_dir():
    bundle = find_bundle(PACKAGE_ROOT)
else:
    if DOWNLOAD_ROOT.exists():
        shutil.rmtree(DOWNLOAD_ROOT)
    DOWNLOAD_ROOT.mkdir(parents=True)
    run(['kaggle', 'datasets', 'download', '-d', DATASET_REF, '-p', DOWNLOAD_ROOT])
    archives = sorted(DOWNLOAD_ROOT.glob('*.zip'))
    if len(archives) != 1:
        raise RuntimeError(f'Expected one downloaded Dataset ZIP, found {archives}')
    extract_outer_package(archives[0], PACKAGE_ROOT)
    bundle = find_bundle(PACKAGE_ROOT)
print('RELATIONAL_PRIMARY_BUNDLE_READY', bundle, flush=True)
launcher = bundle / 'run_relational_primary_bundle.py'
if not launcher.is_file():
    raise FileNotFoundError(f'Bundle lacks launcher: {launcher}')
subprocess.run([sys.executable, str(launcher), '--bundle-root', str(bundle)], check=True)
